# 基于 MindSpore 的 BERT 模型实现对话情绪识别

## 案例介绍

该案例以一个文本情感分类任务为例子来说明BERT模型的整个应用过程。

## 模型简介

BERT全称是来自变换器的双向编码器表征量（Bidirectional Encoder Representations from Transformers），它是Google于2018年末开发并发布的一种新型语言模型。与BERT模型相似的预训练语言模型例如问答、命名实体识别、自然语言推理、文本分类等在许多自然语言处理任务中发挥着重要作用。模型是基于Transformer中的Encoder并加上双向的结构，因此一定要熟练掌握Transformer的Encoder的结构。

BERT模型的主要创新点都在pre-train方法上，即用了Masked Language Model和Next Sentence Prediction两种方法分别捕捉词语和句子级别的representation。

在用Masked Language Model方法训练BERT的时候，随机把语料库中15%的单词做Mask操作。对于这15%的单词做Mask操作分为三种情况：80%的单词直接用[Mask]替换、10%的单词直接替换成另一个新的单词、10%的单词保持不变。

因为涉及到Question Answering (QA) 和 Natural Language Inference (NLI)之类的任务，增加了Next Sentence Prediction预训练任务，目的是让模型理解两个句子之间的联系。与Masked Language Model任务相比，Next Sentence Prediction更简单些，训练的输入是句子A和B，B有一半的几率是A的下一句，输入这两个句子，BERT模型预测B是不是A的下一句。

BERT预训练之后，会保存它的Embedding table和12层Transformer权重（BERT-BASE）或24层Transformer权重（BERT-LARGE）。使用预训练好的BERT模型可以对下游任务进行Fine-tuning，比如：文本分类、相似度判断、阅读理解等。

对话情绪识别（Emotion Detection，简称EmoTect），专注于识别智能对话场景中用户的情绪，针对智能对话场景中的用户文本，自动判断该文本的情绪类别并给出相应的置信度，情绪类型分为积极、消极、中性。 对话情绪识别适用于聊天、客服等多个场景，能够帮助企业更好地把握对话质量、改善产品的用户交互体验，也能分析客服服务质量、降低人工质检成本。

## 环境配置

本案例的运行环境为：

| Python | MindSpore | MindSpore NLP |
| :----- | :-------- | :------------ |
| 3.10    | 2.7.0       | 0.5.1           |

如果你在如[昇思大模型平台](https://xihe.mindspore.cn/training-projects)、[华为云ModelArts](https://www.huaweicloud.com/product/modelarts.html)、[启智社区](https://openi.pcl.ac.cn/)等算力平台的Jupyter在线编程环境中运行本案例，可取消如下代码的注释，进行依赖库安装：

In [ ]:
# !pip install mindspore==2.7.0 mindnlp==0.5.1

其他场景可参考[MindSpore安装指南](https://www.mindspore.cn/install)与[MindSpore NLP安装指南](https://github.com/mindspore-lab/mindnlp?tab=readme-ov-file#installation)进行环境搭建。

In [ ]:
import os
import mindnlp
import mindspore
from datasets import Dataset
from mindspore import nn, context

mindspore.set_context(pynative_synchronize=True)

/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.10.14/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress 

## 数据集

这里提供一份已标注的、经过分词预处理的机器人聊天数据集，来自于百度飞桨团队。数据由两列组成，以制表符（'\t'）分隔，第一列是情绪分类的类别（0表示消极；1表示中性；2表示积极），第二列是以空格分词的中文文本，如下示例，文件为 utf8 编码。

label--text_a

0--谁骂人了？我从来不骂人，我骂的都不是人，你是人吗 ？

1--我有事等会儿就回来和你聊

2--我见到你很高兴谢谢你帮我

这部分主要包括数据集读取，数据格式转换，数据 Tokenize 处理和 pad 操作。

In [ ]:
# download dataset
!wget https://baidu-nlp.bj.bcebos.com/emotion_detection-dataset-1.0.0.tar.gz -O emotion_detection.tar.gz
!tar xvf emotion_detection.tar.gz

--2025-11-25 13:56:41--  https://baidu-nlp.bj.bcebos.com/emotion_detection-dataset-1.0.0.tar.gz
Resolving proxy-notebook.modelarts.com (proxy-notebook.modelarts.com)... 192.168.0.33
Connecting to proxy-notebook.modelarts.com (proxy-notebook.modelarts.com)|192.168.0.33|:8083... connected.
Proxy request sent, awaiting response... 200 OK
Length: 1710581 (1.6M) [application/x-gzip]
Saving to: ‘emotion_detection.tar.gz’

emotion_detection.t 100%[===================>]   1.63M   986KB/s    in 1.7s    

2025-11-25 13:56:43 (986 KB/s) - ‘emotion_detection.tar.gz’ saved [1710581/1710581]

data/
data/test.tsv
data/infer.tsv
data/dev.tsv
data/train.tsv
data/vocab.txt


### 数据加载和数据预处理

具体内容可见下面代码注释。

In [ ]:
# 准备数据集（假设数据格式：标签\t文本）
def load_dataset(path):
    labels, texts = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f.readlines()[1:]:  # 跳过标题行
            if line.strip():
                label, text = line.strip().split('\t')
                labels.append(int(label))
                texts.append(text)
    return {'label': labels, 'text': texts}


In [ ]:
# 加载数据
train_data = load_dataset("data/train.tsv")
val_data = load_dataset("data/dev.tsv")
test_data = load_dataset("data/test.tsv")

# 转换为Hugging Face数据集格式
train_dataset = Dataset.from_dict(train_data)
val_dataset = Dataset.from_dict(val_data)
test_dataset = Dataset.from_dict(test_data)

In [ ]:
print(f"列名: {train_dataset.column_names}")
print(train_dataset[0])


列名: ['label', 'text']
{'label': 1, 'text': '你 的 生日 在 ？ 月 ？ 日'}


In [ ]:
# 加载分词器
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')

In [ ]:
tokenizer.pad_token_id

0

In [ ]:
# 3. 数据预处理
import numpy as np
def preprocess_function(examples, max_seq_len=64):
    """
    对文本进行tokenize处理
    """
    tokenized = tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=max_seq_len,
        return_tensors="pt"
    )
    
    # 将标签转换为int32
    labels = np.array(examples['label'], dtype=np.int32)
    
    return {
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': labels  # 标签会自动保持为整数
    }

tokenized_train = train_dataset.map(preprocess_function, batched=True, batch_size=32, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess_function, batched=True, batch_size=32, remove_columns=val_dataset.column_names)

Map:   2%|▏         | 192/9655 [00:00<00:06, 1557.09 examples/s]

[MS_ALLOC_CONF]Runtime config:  enable_vmm:True  vmm_align_size:2MB


Map: 100%|██████████| 1080/1080 [00:00<00:00, 1699.11 examples/s]


## 数据可视化

为了让大家直观了解到本案例使用的数据，我们数据集信息并打印前3个样本进行展示。

In [ ]:
# 查看数据集信息
print("数据集信息:")
print(f"数据集类型: {type(tokenized_train)}")
print(f"数据集列名: {tokenized_train.column_names}")
print(f"数据集特征: {tokenized_train.features}")
print(f"数据集长度: {len(tokenized_train)}")

# 查看前3个样本
print("\n前3个样本:")
for i in range(min(3, len(tokenized_train))):
    print(f"样本 {i}: {tokenized_train[i]}")

数据集信息:
数据集类型: <class 'datasets.arrow_dataset.Dataset'>
数据集列名: ['input_ids', 'attention_mask', 'labels']
数据集特征: {'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': Value('int32')}
数据集长度: 9655

前3个样本:
样本 0: {'input_ids': [101, 872, 4638, 4495, 3189, 1762, 8043, 3299, 8043, 3189, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': 1}
样本 1: {'input_ids': [101, 1420, 6432, 872, 3221, 671, 702, 3265, 4511, 8024, 3221, 1408, 8043, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

## 模型构建

通过 BertForSequenceClassification 构建用于情感分类的 BERT 模型，加载预训练权重，设置情感三分类的超参数自动构建模型。后面对模型采用自动混合精度操作，提高训练的速度，然后实例化优化器，紧接着实例化评价指标，设置模型训练的权重保存策略，最后就是构建训练器，模型开始训练。

In [ ]:
from transformers import BertForSequenceClassification, BertModel

# set bert config and define parameters for training
model = BertForSequenceClassification.from_pretrained('bert-base-chinese', num_labels=3)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids
    
    if len(predictions.shape) > 1:
        predictions = np.argmax(predictions, axis=-1)

    accuracy = (predictions == labels).mean()
    
    return {"accuracy": float(accuracy)}

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    num_train_epochs=1,
    logging_steps=200,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    args=training_args
)

Detected kernel version 4.19.90, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


## 模型训练

In [ ]:
# start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.216833,0.917593


TrainOutput(global_step=151, training_loss=0.3904628374718672, metrics={'train_runtime': 184.8903, 'train_samples_per_second': 52.22, 'train_steps_per_second': 0.817, 'total_flos': 317545006020480.0, 'train_loss': 0.3904628374718672, 'epoch': 1.0})

## 模型验证

将验证数据集加再进训练好的模型，对数据集进行验证，查看模型在验证数据上面的效果，此处的评价指标为准确率。

In [ ]:
model.npu()  #模型搬运到npu侧
device = model.device  # 获取模型所在的设备
device

In [ ]:
from tqdm import tqdm
import numpy as np

def compute_accuracy(logits, labels):
    preds = np.argmax(logits, axis=-1)
    refs  = labels.asnumpy() if hasattr(labels, "asnumpy") else labels
    return {"accuracy": float((preds == refs).mean())}

def evaluate_fn(model, test_dataset):
    #total = test_dataset.get_dataset_size()
    total = len(test_dataset)
    epoch_acc, step_total = 0.0, 0
    model.eval()

    with tqdm(total=total) as progress_bar:
        for batch in test_dataset.create_dict_iterator():
            label  = batch.pop('labels')
            logits = model(**batch).logits

            acc_dict  = compute_accuracy(logits, label)
            epoch_acc += acc_dict['accuracy']

            step_total += 1
            progress_bar.update(1)
            progress_bar.set_postfix(acc=epoch_acc / step_total)

    return epoch_acc / step_total


In [ ]:
from tqdm import tqdm
import numpy as np
import torch

def compute_accuracy(logits, labels):
    # 使用PyTorch直接计算，避免numpy转换问题
    preds = torch.argmax(logits, dim=-1)
    correct = (preds == labels).float()
    accuracy = correct.mean().item()
    return {"accuracy": accuracy}

def evaluate_fn(model, test_dataset):
    #total = test_dataset.get_dataset_size()
    total = len(test_dataset)
    epoch_acc, step_total = 0.0, 0
    model.eval()
    
    # 创建 DataLoader（因为 test_dataset 应该是 Dataset 对象）
    from torch.utils.data import DataLoader
    from transformers import DataCollatorWithPadding
    
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    dataloader = DataLoader(
        test_dataset,
        batch_size=1,
        collate_fn=data_collator,
        shuffle=False
    )
    
    with torch.no_grad():  # 禁用梯度计算
        with tqdm(total=total) as progress_bar:
            for batch in dataloader:
                batch = {k: v.to(device) for k, v in batch.items()}
                label  = batch.pop('labels')
                logits = model(**batch).logits

                acc_dict  = compute_accuracy(logits, label)
                epoch_acc += acc_dict['accuracy']

                step_total += 1
                progress_bar.update(1)
                progress_bar.set_postfix(acc=epoch_acc / step_total)

    return epoch_acc / step_total

In [ ]:
acc = evaluate_fn(model, tokenized_val)
print(f"Accuracy: {acc}")

100%|██████████| 1080/1080 [01:16<00:00, 14.20it/s, acc=0.918]

Accuracy: 0.9175925925925926


## 模型推理

遍历推理数据集，将结果与标签进行统一展示。

In [ ]:
# 加载数据
dataset_infer = load_dataset("data/infer.tsv")

In [ ]:
dataset_infer

{'label': [1, 0, 1, 1, 0, 1, 2, 1, 1, 0, 1, 1, 0],
 'text': ['我 要 客观',
  '靠 你 真是 说 废话 吗',
  '口嗅 会',
  '每次 是 表妹 带 窝 飞 因为 窝路痴',
  '别说 废话 我 问 你 个 问题',
  '4967 是 新加坡 那 家 银行',
  '是 我 喜欢 兔子',
  '你 写 过 黄山 奇石 吗',
  '一个一个 慢慢来',
  '我 玩 过 这个 一点 都 不 好玩',
  '网上 开发 女孩 的 QQ',
  '背 你 猜 对 了',
  '我 讨厌 你 ， 哼哼 哼 。 。']}

In [ ]:
def predict(text, label=None):
    label_map = {0: "消极", 1: "中性", 2: "积极"}

    #text_tokenized = Tensor([tokenizer(text).input_ids])
    encoded = tokenizer(text, padding='max_length', truncation=True, max_length=64, return_tensors='pt')
    input_ids = encoded['input_ids']
    
    # 移动到设备
    input_ids = input_ids.to(device)
    
    # 模型预测
    with torch.no_grad():
        logits = model(input_ids).logits
        predict_label = logits.argmax(dim=-1).item()  # 使用PyTorch方法
    
    info = f"输入: '{text}', 预测: '{label_map[predict_label]}'"
    if label is not None:
        info += f", 真实标签: '{label_map[label]}'"
        if predict_label == label:
            info += " ✅"
        else:
            info += " ❌"
    print(info)

In [ ]:
from mindspore import Tensor

for label, text in zip(dataset_infer['label'], dataset_infer['text']):
    predict(text, label)

输入: '我 要 客观', 预测: '中性', 真实标签: '中性' ✅
输入: '靠 你 真是 说 废话 吗', 预测: '中性', 真实标签: '消极' ❌
输入: '口嗅 会', 预测: '中性', 真实标签: '中性' ✅
输入: '每次 是 表妹 带 窝 飞 因为 窝路痴', 预测: '中性', 真实标签: '中性' ✅
输入: '别说 废话 我 问 你 个 问题', 预测: '中性', 真实标签: '消极' ❌
输入: '4967 是 新加坡 那 家 银行', 预测: '中性', 真实标签: '中性' ✅
输入: '是 我 喜欢 兔子', 预测: '中性', 真实标签: '积极' ❌
输入: '你 写 过 黄山 奇石 吗', 预测: '中性', 真实标签: '中性' ✅
输入: '一个一个 慢慢来', 预测: '中性', 真实标签: '中性' ✅
输入: '我 玩 过 这个 一点 都 不 好玩', 预测: '中性', 真实标签: '消极' ❌
输入: '网上 开发 女孩 的 QQ', 预测: '中性', 真实标签: '中性' ✅
输入: '背 你 猜 对 了', 预测: '中性', 真实标签: '中性' ✅
输入: '我 讨厌 你 ， 哼哼 哼 。 。', 预测: '中性', 真实标签: '消极' ❌


## 自定义推理数据集

自己输入推理数据，展示模型的泛化能力。

In [ ]:
predict("家人们咱就是说一整个无语住了 绝绝子叠buff")

输入: '家人们咱就是说一整个无语住了 绝绝子叠buff', 预测: '中性'
